# BD-CeNN Fusion (Training + Symbolic Features)
This notebook reuses the dataset management and training workflow from BD-CeNN_Final_Train, 
then swaps in the BD-CeNN architecture + mask-aware symbolic feature extraction inspired by neuro-sym-balanced.


## FINAL (v4) Training + Symbolic Features

- Stage 1: BD-CeNN autoencoder produces binary symbolic latents.
- Stage 2: BD-CeNN reservoir refines symbolic states.
- Stage 3: Symbolic features are derived from refined latents + lesion mask.
- Stage 4: ASP (in a separate notebook) consumes derived predicates.

In [ ]:
# Imports and config
from dataclasses import dataclass
from pathlib import Path
from typing import Tuple, Dict, List, Optional, Sequence
import random
import json as pyjson

import numpy as np
import pandas as pd
from PIL import Image
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import InterpolationMode
import torchvision.transforms.functional as TF

torch.manual_seed(42)
random.seed(42)
np.random.seed(42)

@dataclass
class Config:
    project_root: Path = Path.cwd()
    dataset_root: Path = Path('C:\Users\Admin\Desktop\Desktop\Uniud-Magistrale\SecondYear\LargeProject\Progetto-Medico\Medical_Dataset\Full_Labels\multitask_fully_labeled')
    manifest_csv: Path = dataset_root / 'manifest.csv'
    image_size: Tuple[int, int] = (256, 256)
    batch_size: int = 16

    max_epochs_ae: int = 30
    base_learning_rate: float = 1e-4
    weight_decay: float = 1e-4
    early_stop_patience: int = 5
    early_stop_min_delta: float = 1e-4

    class_names: Tuple[str, ...] = ('MEL', 'NV', 'BCC', 'AKIEC', 'BKL', 'DF', 'VASC')
    malignant_classes: Tuple[str, ...] = ('MEL', 'BCC', 'AKIEC')
    use_binary_labels: bool = True

    latent_channels: int = 64
    bdcenn_depth: int = 3
    reservoir_layers: int = 3
    reservoir_steps: int = 3

    ae_mask_loss_weight: float = 0.2
    ae_density_weight: float = 0.05
    ae_target_density: float = 0.08
    ae_aux_cls_weight: float = 0.2
    ae_aux_hidden: int = 128

    out_dir: Path = Path('C:/Users/Admin/Desktop/Progetto-Medico/Symbolic/BD-CeNN_Fusion/Out_Main')  # [CHANGE] fixed path + relative

    # [CHANGE] predicate binning config
    predicate_quantiles: Tuple[float, float, float, float, float] = (0.1, 0.3, 0.5, 0.7, 0.9)  # [CHANGE] finer bins
    predicate_min_rate: float = 0.01
    predicate_max_rate: float = 0.99

    # [CHANGE] versioned symbolic output (avoid overwriting previous CSVs)
    symbolic_tag: str = "v4"
    symbolic_out: Path = out_dir / f"symbolic_features_{symbolic_tag}"

    # [CHANGE] symbolic feedback (from ASP)
    use_symbolic_feedback: bool = True
    symbolic_feedback_weight: float = 0.2
    feedback_train_csv: Path = out_dir / 'asp_feedback_train.csv'
    feedback_val_csv: Path = out_dir / 'asp_feedback_val.csv'

    skip_training = False

cfg = Config()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

Using device: cuda


In [3]:
cfg.skip_training = True

In [ ]:
# Environment detection: local vs Kaggle
import os
from pathlib import Path

def detect_env():
    if os.environ.get('KAGGLE_KERNEL_RUN_TYPE') or Path('/kaggle/input').exists():
        return 'kaggle'
    return 'local'

ENV = detect_env()
print('Environment:', ENV)

if ENV == 'kaggle':
    cfg.dataset_root = Path('/kaggle/input/multitask-dataset-pseudo-labeled/multitask_fully_labeled')
    cfg.manifest_csv = cfg.dataset_root / 'manifest.csv'
    cfg.out_dir = Path('/kaggle/working/bdcenn_fusion')
    cfg.out_dir.mkdir(parents=True, exist_ok=True)
else:
    cfg.out_dir = Path('C:\Users\Admin\Desktop\Desktop\Uniud-Magistrale\SecondYear\LargeProject\Progetto-Medico\Symbolic\BD-CeNN_Fusion\Out_Main')
    cfg.out_dir.mkdir(parents=True, exist_ok=True)


Environment: local


In [4]:
# Resume helper (safe checkpoint restore)
def try_resume(model, optimizer, ckpt_path: Path):
    if ckpt_path.exists():
        ckpt = torch.load(ckpt_path, map_location=device)
        if isinstance(ckpt, dict) and 'model_state_dict' in ckpt and 'optimizer_state_dict' in ckpt:
            model.load_state_dict(ckpt['model_state_dict'])
            optimizer.load_state_dict(ckpt['optimizer_state_dict'])
            start_epoch = int(ckpt.get('epoch', 0))
            print(f'Resumed from {ckpt_path} @ epoch {start_epoch}')
            return start_epoch
    return 0


In [5]:
# Data pipeline (RGB images + masks)

def load_image_rgb(path: Path, size, train=False):
    img = Image.open(path).convert('RGB')
    if train:
        if random.random() < 0.5:
            img = TF.hflip(img)
        if random.random() < 0.2:
            img = TF.vflip(img)
        if random.random() < 0.3:
            angle = random.uniform(-15, 15)
            img = TF.rotate(img, angle, interpolation=InterpolationMode.BILINEAR)
    img = TF.resize(img, size, interpolation=InterpolationMode.BILINEAR)
    img = TF.to_tensor(img)
    return img


def load_mask(path: Path, size, train=False):
    mask = Image.open(path).convert('L')
    if train:
        if random.random() < 0.5:
            mask = TF.hflip(mask)
        if random.random() < 0.2:
            mask = TF.vflip(mask)
        if random.random() < 0.3:
            angle = random.uniform(-15, 15)
            mask = TF.rotate(mask, angle, interpolation=InterpolationMode.NEAREST)
    mask = TF.resize(mask, size, interpolation=InterpolationMode.NEAREST)
    mask = TF.to_tensor(mask)
    mask = (mask > 0.5).float()
    return mask


class ImageDataset(Dataset):
    def __init__(self, cfg: Config, split: str):
        self.cfg = cfg
        self.split = split
        self.df = pd.read_csv(cfg.manifest_csv)
        self.df = self.df[self.df['split'] == split].reset_index(drop=True)
        if self.df.empty:
            raise RuntimeError(f'No rows for split={split}')

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = self.cfg.dataset_root / row['image']
        mask_path = self.cfg.dataset_root / row['mask']
        img = load_image_rgb(img_path, self.cfg.image_size, train=(self.split=='train'))
        mask = load_mask(mask_path, self.cfg.image_size, train=(self.split=='train'))
        label_vec = row[list(self.cfg.class_names)].to_numpy(dtype=np.float32)
        class_id = int(label_vec.argmax())
        class_name = self.cfg.class_names[class_id]
        label_bin = int(class_name in self.cfg.malignant_classes)
        return {
            'image': img,
            'mask': mask,
            'class_id': torch.tensor(class_id, dtype=torch.long),
            'label_bin': torch.tensor(label_bin, dtype=torch.long),
            'image_id': row['image_id'],
            'image_path': str(img_path),
            'mask_path': str(mask_path),
        }


def create_loaders(cfg: Config):
    datasets = {split: ImageDataset(cfg, split) for split in ('train', 'val', 'test')}
    loaders = {
        split: DataLoader(datasets[split], batch_size=cfg.batch_size, shuffle=(split=='train'), drop_last=(split=='train'))
        for split in ('train', 'val', 'test')
    }
    return datasets, loaders


datasets, loaders = create_loaders(cfg)
print({k: len(v) for k, v in datasets.items()})


{'train': 12609, 'val': 293, 'test': 2512}


In [6]:
# BD-CeNN core (Stage 1 - Symbolic encoding)
class STEBinarize(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        return (x > 0).float()

    @staticmethod
    def backward(ctx, grad_output):
        return grad_output


def ste_binarize(x: torch.Tensor, threshold: float = 0.0) -> torch.Tensor:
    hard = (x >= threshold).float()
    return x + (hard - x).detach()


class BDCeNNCell(nn.Module):
    # Binary, local-rule cellular update (3x3 neighborhood, STE binarization).
    def __init__(self, in_channels: int, out_channels: int, steps: int = 1,
                 alpha: float = 1.0, beta: float = 0.0,
                 trainable: bool = False, learnable_gain: bool = False, learnable_bias: bool = False,
                 detach_dynamics: bool = True):
        super().__init__()
        self.steps = max(1, steps)
        self.detach_dynamics = detach_dynamics
        self.alpha = nn.Parameter(torch.tensor(alpha), requires_grad=learnable_gain)
        self.beta = nn.Parameter(torch.tensor(beta), requires_grad=learnable_bias)
        self.template_state = nn.Parameter(torch.randn(out_channels, in_channels, 3, 3) * 0.1, requires_grad=trainable)
        self.template_input = nn.Parameter(torch.randn(out_channels, in_channels, 3, 3) * 0.1, requires_grad=trainable)
        if not trainable:
            for param in [self.template_state, self.template_input]:
                param.requires_grad = False

    def freeze(self):
        for param in self.parameters():
            param.requires_grad = False

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        driven = F.conv2d(x, self.template_input, padding=1)
        if self.detach_dynamics:
            driven = driven.detach()
        state = ste_binarize(driven)
        for _ in range(self.steps):
            s_in = state.detach() if self.detach_dynamics else state
            neighborhood = F.conv2d(s_in, self.template_state, padding=1)
            combined = self.alpha * s_in + neighborhood + driven + self.beta
            state = ste_binarize(combined)
        return state


class BDCeNNAutoencoder(nn.Module):
    def __init__(self, latent_channels: int = 64, depth: int = 3,
                 trainable_input_lift: bool = False,
                 trainable_templates: bool = False, learnable_gain: bool = False, learnable_bias: bool = False):
        super().__init__()
        self.trainable_input_lift = trainable_input_lift
        self.latent_requires_grad = trainable_input_lift or trainable_templates or learnable_gain or learnable_bias
        self.input_lift = nn.Conv2d(3, latent_channels, kernel_size=3, padding=1, bias=False)
        if not self.trainable_input_lift:
            for param in self.input_lift.parameters():
                param.requires_grad = False
        self.encoder_cells = nn.ModuleList([
            BDCeNNCell(latent_channels, latent_channels, steps=2,
                       trainable=trainable_templates,
                       learnable_gain=learnable_gain,
                       learnable_bias=learnable_bias)
            for _ in range(depth)
        ])
        self.decoder_cells = nn.ModuleList([
            BDCeNNCell(latent_channels, latent_channels, steps=2,
                       trainable=trainable_templates,
                       learnable_gain=learnable_gain,
                       learnable_bias=learnable_bias)
            for _ in range(depth)
        ])
        self.output_head = nn.Conv2d(latent_channels, 3, kernel_size=3, padding=1, bias=True)

    def freeze_dynamics(self) -> None:
        for param in self.input_lift.parameters():
            param.requires_grad = False
        for cell in list(self.encoder_cells) + list(self.decoder_cells):
            cell.freeze()
        for param in self.output_head.parameters():
            param.requires_grad = False
        self.latent_requires_grad = False

    def trainable_parameters(self):
        if self.trainable_input_lift:
            yield from self.input_lift.parameters()
        for cell in list(self.encoder_cells) + list(self.decoder_cells):
            for p in cell.parameters():
                if p.requires_grad:
                    yield p
        for p in self.output_head.parameters():
            if p.requires_grad:
                yield p

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        z = self.input_lift(x)
        for cell in self.encoder_cells:
            z = cell(z)
        return z

    def decode(self, z: torch.Tensor) -> torch.Tensor:
        x = z
        for cell in self.decoder_cells:
            x = cell(x)
        return self.output_head(x)

    def forward(self, x: torch.Tensor):
        z = self.encode(x)
        recon = self.decode(z)
        return recon, z


In [7]:
# Stage 2: BD-CeNN Reservoir (temporal symbolic refinement)
class BDCeNNReservoir(nn.Module):
    def __init__(
        self,
        channels: int = 64,
        depth: int = 3,
        steps: int = 3,
        neighborhood: str = "moore",
        state_coupling: float = 0.6,
        input_coupling: float = 0.6,
        leak: float = 0.0,
        bias: float = -0.2,
        memory_mode: str = "replace",
        memory_decay: float = 0.25,
        inject_input_each_step: bool = True,
        until_converged: bool = False,
        stable_tol: float = 0.0,
        max_converge_steps: int | None = None,
    ) -> None:
        super().__init__()
        self.virtual_steps = max(1, steps)
        self.depth = max(1, depth)
        self.memory_mode = memory_mode
        self.memory_decay = float(memory_decay)
        self.inject_input_each_step = inject_input_each_step
        self.leak = float(leak)
        self.bias = float(bias)
        self.channels = channels
        self.until_converged = until_converged
        self.stable_tol = stable_tol
        self.max_converge_steps = max_converge_steps

        base_kernel = torch.zeros(3, 3)
        if neighborhood.lower() == "moore":
            base_kernel[:] = 1.0
            base_kernel[1, 1] = 0.0
        else:
            base_kernel[0, 1] = base_kernel[1, 0] = base_kernel[1, 2] = base_kernel[2, 1] = 1.0
        state_kernel = torch.zeros(self.depth, channels, 1, 3, 3)
        input_kernel = torch.zeros(self.depth, channels, 1, 3, 3)
        state_kernel[:, :, 0] = state_coupling * base_kernel
        input_kernel[:, :, 0] = input_coupling * base_kernel
        self.register_buffer("state_kernel", state_kernel, persistent=False)
        self.register_buffer("input_kernel", input_kernel, persistent=False)

        self._states: list[torch.Tensor] | None = None

    def reset_state(self) -> None:
        self._states = None

    def _ensure_states(self, latent: torch.Tensor) -> None:
        if self._states is None or len(self._states) != self.depth or self._states[0].shape != latent.shape:
            init = ste_binarize(latent.detach())
            self._states = [init.clone() for _ in range(self.depth)]

    @torch.no_grad()
    def forward(self, latent: torch.Tensor, steps: int | None = None, reset_state: bool = False, until_converged: bool | None = None) -> torch.Tensor:
        if reset_state:
            self.reset_state()
        self._ensure_states(latent)
        k = max(1, steps or self.virtual_steps)
        stop_on_stable = self.until_converged if until_converged is None else until_converged
        max_iters = self.max_converge_steps or k
        for it in range(max_iters):
            drive = latent if self.inject_input_each_step else self._states[0]
            new_states: list[torch.Tensor] = []
            for layer_idx in range(self.depth):
                state = self._states[layer_idx]
                neighbor_feedback = F.conv2d(state, self.state_kernel[layer_idx], padding=1, groups=self.channels)
                external_drive = F.conv2d(drive, self.input_kernel[layer_idx], padding=1, groups=self.channels)
                pre_activation = self.leak * state + neighbor_feedback + external_drive + self.bias
                updated = ste_binarize(pre_activation)

                if self.memory_mode == "or":
                    merged = torch.maximum(state, updated)
                elif self.memory_mode == "ema":
                    merged = ste_binarize(self.memory_decay * state + (1.0 - self.memory_decay) * updated)
                else:
                    merged = updated

                new_states.append(merged)
                drive = merged
            delta = sum((ns - os).abs().sum() for ns, os in zip(new_states, self._states))
            self._states = new_states
            if stop_on_stable and delta.item() <= self.stable_tol:
                break
            if not stop_on_stable and it + 1 >= k:
                break
        return self._states[-1].clone()


In [8]:
# Train / validate AE with early stopping + checkpoints

def train_autoencoder(model, loaders, max_epochs, ckpt_prefix):
    model = model.to(device)

    # Optional auxiliary classifier on pooled latent
    aux_head = None
    cls_loss_fn = None
    if cfg.ae_aux_cls_weight > 0 or (cfg.use_symbolic_feedback and cfg.symbolic_feedback_weight > 0):
        aux_head = nn.Sequential(
            nn.Linear(cfg.latent_channels, cfg.ae_aux_hidden),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(cfg.ae_aux_hidden, 2 if cfg.use_binary_labels else len(cfg.class_names)),
        ).to(device)
        # class weights from binary labels (train split)
        label_bins = loaders['train'].dataset.df.apply(lambda r: 1 if r[list(cfg.class_names)].to_numpy(dtype=float).argmax() in [cfg.class_names.index(c) for c in cfg.positive_classes] else 0, axis=1).to_numpy(dtype=int)
        if label_bins.sum() > 0:
            neg = float((label_bins == 0).sum())
            pos = float((label_bins == 1).sum())
            w_neg = (pos + neg) / (2.0 * max(neg, 1.0))
            w_pos = (pos + neg) / (2.0 * max(pos, 1.0))
            class_weights = torch.tensor([w_neg, w_pos], dtype=torch.float32, device=device)
        else:
            class_weights = None
        cls_loss_fn = nn.CrossEntropyLoss(weight=class_weights)

    params = list(model.parameters()) + (list(aux_head.parameters()) if aux_head is not None else [])
    opt = torch.optim.AdamW(params, lr=cfg.base_learning_rate, weight_decay=cfg.weight_decay)
    recon_loss_fn = nn.MSELoss()

    # [CHANGE] load symbolic feedback maps (image_id -> asp_pred)
    def _load_feedback(path: Path):
        if not path.exists():
            return {}
        df_fb = pd.read_csv(path)
        if 'image_id' not in df_fb.columns or 'asp_pred' not in df_fb.columns:
            return {}
        return {str(r['image_id']): int(r['asp_pred']) for _, r in df_fb.iterrows()}

    use_feedback = bool(cfg.use_symbolic_feedback and cfg.symbolic_feedback_weight > 0)
    feedback_train_map = _load_feedback(cfg.feedback_train_csv) if use_feedback else {}
    feedback_val_map = _load_feedback(cfg.feedback_val_csv) if use_feedback else {}
    ckpt_best = cfg.out_dir / f'{ckpt_prefix}_best_full.pth'
    ckpt_last = cfg.out_dir / f'{ckpt_prefix}_last_full.pth'
    ckpt_path = ckpt_best if ckpt_best.exists() else ckpt_last
    start_epoch = 0
    if ckpt_path.exists():
        ckpt = torch.load(ckpt_path, map_location=device)
        if isinstance(ckpt, dict) and 'model_state_dict' in ckpt:
            try:
                model.load_state_dict(ckpt['model_state_dict'])
                if aux_head is not None and 'aux_head_state_dict' in ckpt:
                    aux_head.load_state_dict(ckpt['aux_head_state_dict'])

                if ckpt_path == ckpt_last and 'optimizer_state_dict' in ckpt:
                    opt.load_state_dict(ckpt['optimizer_state_dict'])
                    start_epoch = int(ckpt.get('epoch', 0))
                    print(f'Resumed {ckpt_prefix} from epoch {start_epoch}')
                else:
                    print(f'Loaded best checkpoint for {ckpt_prefix} (fine-tune from epoch 0)')
            except Exception as e:
                print('Skip resume: checkpoint mismatch', e)

    best_val = 1e9
    no_improve = 0
    history = []

    for epoch in range(start_epoch, max_epochs):
        model.train()
        if aux_head is not None:
            aux_head.train()

        tr_total = tr_recon = tr_mask = tr_den = tr_cls = tr_sym = 0.0
        for batch in loaders['train']:
            images = batch['image'].to(device)
            masks = batch['mask'].to(device)
            labels = batch['label_bin' if cfg.use_binary_labels else 'class_id'].to(device)

            opt.zero_grad(set_to_none=True)
            recon, latent = model(images)

            loss_recon = recon_loss_fn(recon, images)
            loss_total = loss_recon

            loss_mask = torch.tensor(0.0, device=device)
            if cfg.ae_mask_loss_weight > 0:
                mask_down = F.interpolate(masks, size=latent.shape[-2:], mode='bilinear', align_corners=False)
                latent_map = latent.mean(dim=1, keepdim=True)
                loss_mask = F.mse_loss(latent_map, mask_down)
                loss_total = loss_total + cfg.ae_mask_loss_weight * loss_mask

            loss_den = torch.tensor(0.0, device=device)
            if cfg.ae_density_weight > 0:
                density = latent.mean()
                loss_den = (density - cfg.ae_target_density) ** 2
                loss_total = loss_total + cfg.ae_density_weight * loss_den

            loss_cls = torch.tensor(0.0, device=device)
            loss_sym = torch.tensor(0.0, device=device)
            if aux_head is not None:
                pooled = latent.mean(dim=(2, 3))
                logits = aux_head(pooled)
                loss_cls = cls_loss_fn(logits, labels)
                loss_total = loss_total + cfg.ae_aux_cls_weight * loss_cls

                # [CHANGE] symbolic feedback loss (ASP-derived)
                if use_feedback and feedback_train_map:
                    ids = batch['image_id']
                    sym_targets = [feedback_train_map.get(str(i), -1) for i in ids]
                    sym_targets = torch.tensor(sym_targets, device=device)
                    mask = sym_targets >= 0
                    if mask.any():
                        loss_sym = cls_loss_fn(logits[mask], sym_targets[mask])
                        loss_total = loss_total + cfg.symbolic_feedback_weight * loss_sym

            loss_total.backward()
            opt.step()

            tr_total += loss_total.item()
            tr_recon += loss_recon.item()
            tr_mask += loss_mask.item()
            tr_den += loss_den.item()
            tr_cls += loss_cls.item()
            tr_sym += loss_sym.item()

        # Validation
        model.eval()
        if aux_head is not None:
            aux_head.eval()
        va_total = va_recon = va_mask = va_den = va_cls = va_sym = 0.0
        with torch.no_grad():
            for batch in loaders['val']:
                images = batch['image'].to(device)
                masks = batch['mask'].to(device)
                labels = batch['label_bin' if cfg.use_binary_labels else 'class_id'].to(device)
                recon, latent = model(images)

                loss_recon = recon_loss_fn(recon, images)
                loss_total = loss_recon

                loss_mask = torch.tensor(0.0, device=device)
                if cfg.ae_mask_loss_weight > 0:
                    mask_down = F.interpolate(masks, size=latent.shape[-2:], mode='bilinear', align_corners=False)
                    latent_map = latent.mean(dim=1, keepdim=True)
                    loss_mask = F.mse_loss(latent_map, mask_down)
                    loss_total = loss_total + cfg.ae_mask_loss_weight * loss_mask

                loss_den = torch.tensor(0.0, device=device)
                if cfg.ae_density_weight > 0:
                    density = latent.mean()
                    loss_den = (density - cfg.ae_target_density) ** 2
                    loss_total = loss_total + cfg.ae_density_weight * loss_den

                loss_cls = torch.tensor(0.0, device=device)
                loss_sym = torch.tensor(0.0, device=device)
                if aux_head is not None:
                    pooled = latent.mean(dim=(2, 3))
                    logits = aux_head(pooled)
                    loss_cls = cls_loss_fn(logits, labels)
                    loss_total = loss_total + cfg.ae_aux_cls_weight * loss_cls

                    # [CHANGE] symbolic feedback loss (ASP-derived)
                    if use_feedback and feedback_val_map:
                        ids = batch['image_id']
                        sym_targets = [feedback_val_map.get(str(i), -1) for i in ids]
                        sym_targets = torch.tensor(sym_targets, device=device)
                        mask = sym_targets >= 0
                        if mask.any():
                            loss_sym = cls_loss_fn(logits[mask], sym_targets[mask])
                            loss_total = loss_total + cfg.symbolic_feedback_weight * loss_sym

                va_total += loss_total.item()
                va_recon += loss_recon.item()
                va_mask += loss_mask.item()
                va_den += loss_den.item()
                va_cls += loss_cls.item()
                va_sym += loss_sym.item()

        tr_total /= max(1, len(loaders['train']))
        tr_recon /= max(1, len(loaders['train']))
        tr_mask /= max(1, len(loaders['train']))
        tr_den /= max(1, len(loaders['train']))
        tr_cls /= max(1, len(loaders['train']))
        tr_sym /= max(1, len(loaders['train']))

        va_total /= max(1, len(loaders['val']))
        va_recon /= max(1, len(loaders['val']))
        va_mask /= max(1, len(loaders['val']))
        va_den /= max(1, len(loaders['val']))
        va_cls /= max(1, len(loaders['val']))
        va_sym /= max(1, len(loaders['val']))

        print(
            f"{ckpt_prefix} Epoch {epoch+1:02d}: "
            f"train_total={tr_total:.4f} train_recon={tr_recon:.4f} train_mask={tr_mask:.4f} "
            f"train_den={tr_den:.4f} train_cls={tr_cls:.4f} train_sym={tr_sym:.4f} | "
            f"val_total={va_total:.4f} val_recon={va_recon:.4f} val_mask={va_mask:.4f} "
            f"val_den={va_den:.4f} val_cls={va_cls:.4f} val_sym={va_sym:.4f}"
        )

        history.append({
            'epoch': epoch + 1,
            'train_total': tr_total,
            'train_recon': tr_recon,
            'train_mask': tr_mask,
            'train_den': tr_den,
            'train_cls': tr_cls,
            'train_sym': tr_sym,
            'val_total': va_total,
            'val_recon': va_recon,
            'val_mask': va_mask,
            'val_den': va_den,
            'val_cls': va_cls,
            'val_sym': va_sym,
        })

        # save best
        if va_total < best_val - cfg.early_stop_min_delta:
            best_val = va_total
            torch.save(model.state_dict(), cfg.out_dir / f'{ckpt_prefix}_best.pth')
            torch.save({
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': opt.state_dict(),
                'epoch': epoch + 1,
                'aux_head_state_dict': aux_head.state_dict() if aux_head is not None else None,
            }, cfg.out_dir / f'{ckpt_prefix}_best_full.pth')
            no_improve = 0
        else:
            no_improve += 1

        torch.save({
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': opt.state_dict(),
            'epoch': epoch + 1,
            'aux_head_state_dict': aux_head.state_dict() if aux_head is not None else None,
        }, ckpt_last)

        if no_improve >= cfg.early_stop_patience:
            print(f'{ckpt_prefix} Early stopping at epoch {epoch+1}')
            break

    hist_path = cfg.out_dir / f'{ckpt_prefix}_history.json'
    with hist_path.open('w', encoding='utf-8') as f:
        pyjson.dump(history, f, indent=2)
    print('Saved history:', hist_path)
    return model


In [9]:
# Train BD-CeNN autoencoder (fusion)
model = BDCeNNAutoencoder(
    latent_channels=cfg.latent_channels,
    depth=cfg.bdcenn_depth,
    trainable_input_lift=True,
    trainable_templates=True,
    learnable_gain=True,
    learnable_bias=True,
)

if not cfg.skip_training:
    model = train_autoencoder(model, loaders, cfg.max_epochs_ae, 'bdcenn_fusion')


In [10]:
# Quick reconstruction check (test split)

model = model.to(device)
model.eval()

def eval_recon(model, loader):
    model.eval()
    loss_fn = nn.MSELoss()
    total = 0.0
    with torch.no_grad():
        for batch in loader:
            images = batch['image'].to(device)
            recon, _ = model(images)
            total += loss_fn(recon, images).item()
    return total / max(1, len(loader))

print('Fusion AE test recon loss:', eval_recon(model, loaders['test']))


Fusion AE test recon loss: 0.7132917953904268


In [11]:
# Load trained model for predicate extraction
ckpt = Path('C:/Users/Admin/Desktop/Progetto-Medico/Symbolic/BD-CeNN_Fusion/Out_Main/bdcenn_fusion_best.pth')
model = BDCeNNAutoencoder(
    latent_channels=cfg.latent_channels,
    depth=cfg.bdcenn_depth,
    trainable_input_lift=True,
    trainable_templates=True,
    learnable_gain=True,
    learnable_bias=True,
).to(device)

model.load_state_dict(torch.load(ckpt, map_location=device))
model.eval()
print('Loaded checkpoint:', ckpt)

Loaded checkpoint: C:\Users\Admin\Desktop\Progetto-Medico\Symbolic\BD-CeNN_Fusion\Out_Main\bdcenn_fusion_best.pth


In [13]:
# Extract symbolic features with reservoir + masks and save CSVs
# NOTE: Updated with richer predicates + quantile binning. [CHANGE]

SYMBOL_OUT = cfg.symbolic_out  # [CHANGE] versioned output folder
SYMBOL_OUT.mkdir(parents=True, exist_ok=True)

print(SYMBOL_OUT)

# --- Color features (expanded, HSV bins + entropy) [CHANGE] ---

def _load_rgb_uint8(path: str) -> np.ndarray:
    bgr = cv2.imread(path, cv2.IMREAD_COLOR)
    if bgr is None:
        raise FileNotFoundError(path)
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)


def _load_mask_bool(path: Optional[str]) -> Optional[np.ndarray]:
    if not path:
        return None
    mask = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if mask is None:
        return None
    mask_bool = mask > 127
    if not mask_bool.any():
        return None
    return mask_bool


def compute_hsv_color_concepts(image_path: str, mask_path: Optional[str] = None, hue_bins: int = 16) -> Dict[str, float]:  # [CHANGE] more hue bins
    rgb = _load_rgb_uint8(image_path)
    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV).astype(np.float32)
    lesion_mask = _load_mask_bool(mask_path)
    if lesion_mask is not None:
        pixels = hsv[lesion_mask]
    else:
        pixels = hsv.reshape(-1, 3)
    if pixels.size == 0:
        pixels = hsv.reshape(-1, 3)
    h = pixels[:, 0] / 180.0
    s = pixels[:, 1] / 255.0
    v = pixels[:, 2] / 255.0
    hue_edges = np.linspace(0.0, 1.0, hue_bins + 1)
    hue_hist, _ = np.histogram(h, bins=hue_edges)
    hue_hist = hue_hist.astype(np.float32) / (hue_hist.sum() + 1e-6)

    hue_variety = float((hue_hist > 0.08).sum())  # [CHANGE] lower threshold for richer signals
    saturation_mean = float(s.mean())
    value_mean = float(v.mean())
    red_ratio = float(((h <= 0.08) | (h >= 0.92)).mean())

    blue_ratio = float(((h >= 0.55) & (h <= 0.70)).mean())
    saturation_std = float(s.std())
    value_std = float(v.std())
    hue_entropy = float(-np.sum(hue_hist * np.log(hue_hist + 1e-6)))  # [CHANGE]

    stats = {
        'value_mean': value_mean,
        'saturation_mean': saturation_mean,
        'hue_variety': hue_variety,
        'red_ratio': red_ratio,
        'blue_ratio': blue_ratio,
        'saturation_std': saturation_std,
        'value_std': value_std,
        'hue_entropy': hue_entropy,  # [CHANGE]
    }

    # expose raw hue-bin proportions (numeric, will be quantile-binned) [CHANGE]
    for b in range(hue_bins):
        stats[f'hue_bin_{b}'] = float(hue_hist[b])

    stats['color_dark'] = value_mean < 0.45
    stats['color_variegated'] = hue_variety >= 2
    stats['color_reddish'] = red_ratio >= 0.15
    stats['color_bluish'] = blue_ratio >= 0.05
    stats['color_irregular'] = saturation_std >= 0.15 or value_std >= 0.15

    return stats


# --- Latent features (expanded shape/texture) [CHANGE] ---

def compute_latent_features(latent: torch.Tensor, mask: torch.Tensor) -> Dict[str, float]:
    latent_np = latent.detach().cpu().numpy()
    mask_np = mask.detach().cpu().numpy()

    binary = (latent_np > 0.5).astype(np.float32)
    num_channels, h, w = binary.shape
    total_pixels = float(h * w)

    latent_density = float(binary.mean())
    channel_activations = binary.mean(axis=(1, 2))
    area_ratio = float(channel_activations.var())

    mean_activation_map = binary.mean(axis=0)

    # symmetry (simple)
    if mean_activation_map.sum() > 0:
        horiz = np.abs(mean_activation_map - np.fliplr(mean_activation_map)).mean()
        vert = np.abs(mean_activation_map - np.flipud(mean_activation_map)).mean()
        symmetry_score = float(1.0 - 0.5 * (horiz + vert))
    else:
        symmetry_score = 0.0

    # compactness
    if mean_activation_map.sum() > 0:
        yy, xx = np.mgrid[:h, :w]
        total_mass = mean_activation_map.sum()
        cy = (yy * mean_activation_map).sum() / total_mass
        cx = (xx * mean_activation_map).sum() / total_mass
        distances = np.sqrt((yy - cy) ** 2 + (xx - cx) ** 2)
        mean_distance = (distances * mean_activation_map).sum() / total_mass
        max_possible_distance = np.sqrt(h ** 2 + w ** 2) / 2
        compactness = 1.0 - (mean_distance / max_possible_distance)
    else:
        compactness = 0.0

    # channel diversity (entropy)
    probs = channel_activations / (channel_activations.sum() + 1e-6)
    probs = probs[probs > 1e-6]
    if probs.size > 0:
        entropy = -np.sum(probs * np.log(probs + 1e-6))
        max_entropy = np.log(num_channels)
        channel_diversity = float(entropy / max_entropy) if max_entropy > 0 else 0.0
    else:
        channel_diversity = 0.0

    # center-periphery contrast (simple gradient)
    gy, gx = np.gradient(mean_activation_map)
    center_periphery_contrast = float(np.sqrt(gx**2 + gy**2).mean())

    # [CHANGE] border irregularity (perimeter/area proxy)
    binary_map = (mean_activation_map > 0.3).astype(np.uint8)
    if binary_map.sum() > 0:
        kernel = np.ones((3, 3), np.uint8)
        eroded = cv2.erode(binary_map, kernel, iterations=1)
        border = (binary_map - eroded).clip(min=0)
        perimeter = float(border.sum())
        area = float(binary_map.sum())
        border_irregularity = 1.0 - min(1.0, (4 * np.pi * area) / (perimeter ** 2 + 1e-6)) if perimeter > 0 else 0.0
    else:
        border_irregularity = 0.0

    # [CHANGE] texture heterogeneity (patch variance)
    patch = max(2, h // 4)
    patch_vals = []
    for i in range(0, h - patch + 1, patch):
        for j in range(0, w - patch + 1, patch):
            patch_vals.append(mean_activation_map[i:i+patch, j:j+patch].mean())
    texture_heterogeneity = float(np.var(patch_vals)) if patch_vals else 0.0

    # region stats from mask
    mask_bin = (mask_np > 0.5).astype(np.float32)
    if mask_bin.ndim == 3:
        mask_bin = mask_bin[0]
    region_pixels = mask_bin.sum()
    background_pixels = total_pixels - region_pixels

    region_mean = float((binary * mask_bin).sum() / (region_pixels + 1e-6)) if region_pixels > 0 else 0.0
    background_mean = float((binary * (1 - mask_bin)).sum() / (background_pixels + 1e-6)) if background_pixels > 0 else 0.0
    region_area = float(region_pixels / total_pixels) if total_pixels > 0 else 0.0
    region_bg_contrast = float(region_mean - background_mean)

    if region_pixels > 1:
        region_vals = binary[:, mask_bin.astype(bool)]
        region_coherence = float(1.0 / (region_vals.std() + 1e-6))
    else:
        region_coherence = 0.0

    region_active_channels = float((channel_activations > 0.1).sum())

    # [CHANGE] mask compactness (shape of lesion mask)
    mask_u8 = (mask_bin > 0.5).astype(np.uint8)
    if mask_u8.sum() > 0:
        kernel = np.ones((3, 3), np.uint8)
        eroded = cv2.erode(mask_u8, kernel, iterations=1)
        border = (mask_u8 - eroded).clip(min=0)
        per = float(border.sum())
        area = float(mask_u8.sum())
        mask_compactness = min(1.0, (4 * np.pi * area) / (per ** 2 + 1e-6)) if per > 0 else 0.0
    else:
        per = 0.0
        border = np.zeros_like(mask_u8, dtype=np.float32)
        mask_compactness = 0.0

    # [CHANGE] mask asymmetry (clinical proxy)
    if mask_u8.sum() > 0:
        m = mask_u8.astype(np.float32)
        asym_h = np.abs(m - np.fliplr(m)).mean()
        asym_v = np.abs(m - np.flipud(m)).mean()
        mask_asymmetry = float(0.5 * (asym_h + asym_v))
    else:
        mask_asymmetry = 0.0

    # [CHANGE] mask eccentricity + solidity + border_std
    coords = np.column_stack(np.where(mask_u8 > 0))
    if coords.shape[0] >= 3:
        cov = np.cov(coords.T)
        eigvals = np.linalg.eigvalsh(cov)
        if eigvals.max() > 0:
            mask_eccentricity = float(1.0 - (eigvals.min() / eigvals.max()))
        else:
            mask_eccentricity = 0.0
    else:
        mask_eccentricity = 0.0

    # solidity approximation via convex hull area
    try:
        import cv2 as _cv2
        cnts, _ = _cv2.findContours(mask_u8, _cv2.RETR_EXTERNAL, _cv2.CHAIN_APPROX_SIMPLE)
        if cnts:
            cnt = max(cnts, key=_cv2.contourArea)
            hull = _cv2.convexHull(cnt)
            hull_area = float(_cv2.contourArea(hull))
            mask_area = float(_cv2.contourArea(cnt))
            mask_solidity = (mask_area / hull_area) if hull_area > 0 else 0.0
        else:
            mask_solidity = 0.0
    except Exception:
        mask_solidity = 0.0

    # border std as rough irregularity
    if per > 0:
        mask_border_std = float(border.std())
    else:
        mask_border_std = 0.0

    return {
        'latent_density': latent_density,
        'area_ratio': area_ratio,
        'symmetry_score': symmetry_score,
        'compactness': compactness,
        'channel_diversity': channel_diversity,
        'center_periphery_contrast': center_periphery_contrast,
        'border_irregularity': border_irregularity,  # [CHANGE]
        'texture_heterogeneity': texture_heterogeneity,  # [CHANGE]
        'region_mean': region_mean,
        'background_mean': background_mean,
        'region_bg_contrast': region_bg_contrast,
        'region_area': region_area,
        'region_coherence': region_coherence,
        'region_active_channels': region_active_channels,
        'mask_compactness': mask_compactness,  # [CHANGE]
        'mask_eccentricity': mask_eccentricity,  # [CHANGE]
        'mask_solidity': mask_solidity,  # [CHANGE]
        'mask_border_std': mask_border_std,  # [CHANGE]
        'mask_asymmetry': mask_asymmetry,  # [CHANGE]
    }


# --- Build feature tables ---

def build_symbolic_dataframe(model, loader, split: str) -> pd.DataFrame:
    model.eval()
    reservoir = BDCeNNReservoir(
        channels=cfg.latent_channels,
        depth=cfg.reservoir_layers,
        steps=cfg.reservoir_steps,
    ).to(device)
    rows = []
    with torch.no_grad():
        for batch in loader:
            images = batch['image'].to(device)
            masks = batch['mask'].to(device)
            class_ids = batch['class_id'].cpu().numpy().tolist()
            label_bins = batch['label_bin'].cpu().numpy().tolist()
            image_ids = batch['image_id']
            image_paths = batch['image_path']
            mask_paths = batch['mask_path']

            recon, latent = model(images)
            reservoir.reset_state()
            refined = reservoir(latent)

            for i in range(refined.size(0)):
                feats = compute_latent_features(refined[i], masks[i])
                # color features from original image
                color_stats = compute_hsv_color_concepts(image_paths[i], mask_paths[i])
                feats.update(color_stats)
                feats.update({
                    'image_id': image_ids[i],
                    'class_id': int(class_ids[i]),
                    'label_bin': int(label_bins[i]),
                    'split': split,
                })
                rows.append(feats)

    return pd.DataFrame(rows)


# --- Quantile binning (multi-bin predicates) [CHANGE] ---

def build_quantile_map(train_df: pd.DataFrame, feature_cols: List[str], quantiles: Tuple[float, float, float, float, float]):  # [CHANGE] 5 bins
    qmap = {}
    for feat in feature_cols:
        vals = train_df[feat].to_numpy(dtype=float)
        qmap[feat] = np.nanquantile(vals, quantiles)
    return qmap


def apply_quantile_bins(df: pd.DataFrame, qmap: Dict[str, np.ndarray], keep_raw: List[str]):
    out = df.copy()
    for feat, q in qmap.items():
        q1, q2, q3, q4, q5 = q
        out[f'{feat}_q1'] = (out[feat] <= q1).astype(int)
        out[f'{feat}_q2'] = ((out[feat] > q1) & (out[feat] <= q2)).astype(int)
        out[f'{feat}_q3'] = ((out[feat] > q2) & (out[feat] <= q3)).astype(int)
        out[f'{feat}_q4'] = ((out[feat] > q3) & (out[feat] <= q4)).astype(int)
        out[f'{feat}_q5'] = ((out[feat] > q4) & (out[feat] <= q5)).astype(int)
        out[f'{feat}_q6'] = (out[feat] > q5).astype(int)

    # keep already-binary color flags (do not re-binarize)
    for col in keep_raw:
        if col in out.columns:
            out[col] = out[col].astype(int)
    return out


# --- Clinical predicate layer (ABCD-style) [CHANGE] ---

def add_clinical_predicates(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    def _any(cols: List[str]) -> pd.Series:
        cols = [c for c in cols if c in out.columns]
        if not cols:
            return pd.Series(0, index=out.index)
        acc = out[cols[0]].astype(int)
        for c in cols[1:]:
            acc = acc | out[c].astype(int)
        return acc

    # A: Asymmetry
    out['clin_asymmetry'] = _any([
        'symmetry_score_q1', 'symmetry_score_q2',
        'mask_asymmetry_q5', 'mask_asymmetry_q6',
        'mask_eccentricity_q5', 'mask_eccentricity_q6',
    ])

    # B: Border irregularity
    out['clin_border_irregular'] = _any([
        'border_irregularity_q5', 'border_irregularity_q6',
        'mask_border_std_q5', 'mask_border_std_q6',
        'mask_compactness_q1', 'mask_compactness_q2',
        'mask_solidity_q1', 'mask_solidity_q2',
    ])

    # C: Color variegation
    out['clin_color_variegation'] = _any([
        'hue_entropy_q5', 'hue_entropy_q6',
        'hue_variety_q5', 'hue_variety_q6',
        'saturation_std_q5', 'saturation_std_q6',
        'value_std_q5', 'value_std_q6',
        'color_variegated', 'color_irregular',
    ])

    # D: Diameter / size proxy
    out['clin_large_lesion'] = _any(['region_area_q5', 'region_area_q6'])

    # Additional clinical color cues
    out['clin_dark'] = _any(['value_mean_q1', 'value_mean_q2', 'color_dark'])
    out['clin_reddish'] = _any(['red_ratio_q5', 'red_ratio_q6', 'color_reddish'])
    out['clin_bluish'] = _any(['blue_ratio_q5', 'blue_ratio_q6', 'color_bluish'])

    # Texture / structural heterogeneity
    out['clin_texture_rough'] = _any(['texture_heterogeneity_q5', 'texture_heterogeneity_q6'])

    return out


# Build feature tables per split
train_df = build_symbolic_dataframe(model, loaders['train'], 'train')
val_df = build_symbolic_dataframe(model, loaders['val'], 'val')
test_df = build_symbolic_dataframe(model, loaders['test'], 'test')

print("Symbolic Dataframes Built")

feature_cols = [
    'latent_density', 'area_ratio', 'symmetry_score', 'compactness',
    'channel_diversity', 'center_periphery_contrast', 'border_irregularity', 'texture_heterogeneity',
    'region_mean', 'background_mean', 'region_bg_contrast', 'region_area',
    'region_coherence', 'region_active_channels', 'mask_compactness',
    'mask_eccentricity', 'mask_solidity', 'mask_border_std', 'mask_asymmetry',
    'value_mean', 'saturation_mean', 'hue_variety', 'red_ratio', 'blue_ratio',
    'saturation_std', 'value_std', 'hue_entropy'
]

# include hue_bin_* numeric features [CHANGE]
for b in range(16):  # [CHANGE]
    feature_cols.append(f'hue_bin_{b}')

raw_predicates = ['color_dark', 'color_variegated', 'color_reddish', 'color_bluish', 'color_irregular']

print("Applying Quartiles")
qmap = build_quantile_map(train_df, feature_cols, cfg.predicate_quantiles)
train_pred = apply_quantile_bins(train_df, qmap, raw_predicates)
val_pred = apply_quantile_bins(val_df, qmap, raw_predicates)
test_pred = apply_quantile_bins(test_df, qmap, raw_predicates)

print("Adding Clinical Predicates")
# [CHANGE] clinical ABCD-style predicates (binary)
train_pred = add_clinical_predicates(train_pred)
val_pred = add_clinical_predicates(val_pred)
test_pred = add_clinical_predicates(test_pred)

print("Dropping Always ON/OFF Predicates")
# Drop predicates that are always-on or always-off based on train activation rates
rate_hi = cfg.predicate_max_rate  # [CHANGE]
rate_lo = cfg.predicate_min_rate  # [CHANGE]

pred_cols = [c for c in train_pred.columns if c not in ['image_id','class_id','label_bin','split']]

rates = train_pred[pred_cols].mean()
keep_cols = [c for c in pred_cols if (rates[c] < rate_hi and rates[c] > rate_lo)]

print(f'Dropping {len(pred_cols) - len(keep_cols)} predicates (always-on/off). Keeping {len(keep_cols)}.')


train_pred = train_pred[['image_id','class_id','label_bin','split'] + keep_cols]
val_pred = val_pred[['image_id','class_id','label_bin','split'] + keep_cols]
test_pred = test_pred[['image_id','class_id','label_bin','split'] + keep_cols]

print("Saving Outputs")
# Save outputs (versioned)
train_df.to_csv(SYMBOL_OUT / 'features_train.csv', index=False)
val_df.to_csv(SYMBOL_OUT / 'features_val.csv', index=False)
test_df.to_csv(SYMBOL_OUT / 'features_test.csv', index=False)

train_pred.to_csv(SYMBOL_OUT / 'predicates_train.csv', index=False)
val_pred.to_csv(SYMBOL_OUT / 'predicates_val.csv', index=False)
test_pred.to_csv(SYMBOL_OUT / 'predicates_test.csv', index=False)

# [CHANGE] make quantiles JSON-serializable
qmap_json = {k: [float(x) for x in v] for k, v in qmap.items()}
with (SYMBOL_OUT / 'feature_thresholds.json').open('w', encoding='utf-8') as f:
    pyjson.dump({'quantiles': qmap_json, 'predicate_quantiles': cfg.predicate_quantiles}, f, indent=2)

print('Saved symbolic features and predicates to:', SYMBOL_OUT)


C:\Users\Admin\Desktop\Progetto-Medico\Symbolic\BD-CeNN_Fusion\Out_Main\symbolic_features_v4
Symbolic Dataframes Built
Applying Quartiles


C:\Users\Admin\AppData\Local\Temp\ipykernel_4656\862319862.py:305: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[f'{feat}_q1'] = (out[feat] <= q1).astype(int)
C:\Users\Admin\AppData\Local\Temp\ipykernel_4656\862319862.py:306: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[f'{feat}_q2'] = ((out[feat] > q1) & (out[feat] <= q2)).astype(int)
C:\Users\Admin\AppData\Local\Temp\ipykernel_4656\862319862.py:307: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, wh

Adding Clinical Predicates
Dropping Always ON/OFF Predicates
Dropping 156 predicates (always-on/off). Keeping 158.
Saving Outputs
Saved symbolic features and predicates to: C:\Users\Admin\Desktop\Progetto-Medico\Symbolic\BD-CeNN_Fusion\Out_Main\symbolic_features_v4
